# Task 1.3 — What the Paper Claims to Improve
**Paper:** Efficient Additive Kernels via Explicit Feature Maps  
**Student:** Nikhil Raj | Roll No: 230080

---

## 1. Main Baseline Methods

The paper compares against two primary baselines:

**Baseline A — Exact Nonlinear Kernel SVM (LIBSVM):**  
A standard SVM trained using the exact χ², intersection, or Hellinger's kernel with the LIBSVM solver. This is the gold-standard in terms of accuracy but is computationally expensive. The paper refers to this as the "exact" method throughout Tables 1–3.

**Baseline B — Maji and Berg (MB) Approximation [Reference 2 in paper]:**  
Maji and Berg (ICCV 2009) proposed a sparse, high-dimensional explicit feature map specifically for the *intersection kernel only*, based on the Heaviside step function. This is the paper's closest methodological predecessor for data-independent feature maps and is directly compared in Section 6 and Table 2.

A third comparison method is **addKPCA** (Perronnin et al. [1]), a data-dependent Nyström-based approximation applied dimension-by-dimension, which serves as the benchmark for data-dependent approaches.

---

## 2. Limitation of the Baseline Identified by the Paper

**For Baseline A (Exact Kernel SVM):**  
The paper identifies a fundamental scalability problem. Evaluating an exact nonlinear SVM requires computing the kernel K(**x**, **xᵢ**) between the test point and every support vector, making inference O(n_sv × D) — which scales linearly with the number of support vectors retained from training. Training is even worse at O(n²) to O(n³). As the paper states in the Introduction: "Since in most cases evaluating the inner product ⟨**w**,**x**⟩ is about as costly as evaluating the kernel K(**x**,**xᵢ**), this makes the evaluation of the non-linear SVM N times slower than the linear one." For large-scale datasets, this makes exact kernel SVMs entirely impractical.

**For Baseline B (Maji and Berg):**  
The MB approximation is limited to the *intersection kernel only* and cannot be applied to χ², Hellinger's, or JS kernels. Furthermore, it operates in a high-dimensional sparse space and requires modifying the SVM solver's regularizer (replacing **w**ᵀ**w** with **w**ᵀH**w**, where H = DDᵀ), which means it is incompatible with off-the-shelf solvers like LIBLINEAR and PEGASOS without algorithmic changes.

---

## 3. How the Proposed Method Overcomes This Limitation

The paper introduces a unified theoretical framework — the *homogeneous kernel map* — that derives a closed-form, data-independent, finite-dimensional feature map for the entire class of additive homogeneous kernels (not just one kernel), using 1D Fourier analysis of the kernel signature. By transforming the data once using this map, the expensive nonlinear kernel SVM is replaced by a plain linear SVM, making training O(n × (2n+1)D) and inference O((2n+1)D) — a fixed cost independent of the training set size. The approximation error decays exponentially with the approximation order n for smooth kernels (χ², JS), and the map requires no training data, no solver modification, and works with any standard linear SVM library unchanged.

---

## 4. One Condition Where the Paper's Method Would NOT Outperform the Baseline

**Scenario: Data with signed (negative) or non-histogram-type features where the RBF kernel is the appropriate similarity measure.**

The homogeneous kernel map strictly requires non-negative, histogram-type input features. If the input data is not naturally a histogram — for example, dense continuous measurements like sensor readings, PCA-projected embeddings, or z-scored tabular data — then the χ², intersection, and Hellinger's kernels are not the semantically appropriate choice of kernel for that data type. In this case, the Gaussian RBF kernel (which is designed for Euclidean feature spaces) would typically outperform additive kernels regardless of approximation quality.

The paper itself demonstrates this: Table 4 (DC Pedestrians, Generalised Gaussian RBF results) shows that exact RBF kernels achieve a 20–30% relative EER reduction compared to the homogeneous additive kernels — meaning the additive kernel approximation, however accurate, is simply working with an inferior kernel for that feature representation. More fundamentally, applying the homogeneous kernel map to features with negative values (which a Standard-Scaler-preprocessed dataset would have) causes the feature map to produce NaN values due to the log(xᵢ) term, making the method fail completely while the exact RBF kernel SVM would still function correctly.

This is not a failure of the approximation quality, but a failure of the method's fundamental applicability assumption — the kernel family it approximates is inappropriate for non-histogram data. The baseline (exact RBF-SVM) would outperform in this regime because it is designed for exactly this type of data.
